# 🔴 PIXEL Dual-Rover + Satellite — Unified GRPO Training

Train **Llama 3.1 8B** (4-bit LoRA via Unsloth) to control:
- 🚀 **Mars Rover** — Deep autonomy, comm delay, dust storms
- 🌕 **Moon Rover** — Day/night survival, speed optimization
- 🛰️ **Satellite Network** — Multi-agent coordination (SAT-A/B/C)

| Component | Value |
|-----------|-------|
| Model | unsloth/meta-llama-3.1-8b-instruct (4-bit LoRA) |
| Framework | HuggingFace TRL GRPOTrainer |
| Environment | PIXEL OpenEnv (Mars + Moon + Satellite) |
| Reward | Composable rubric (science + survival + efficiency + coordination) |

## Section 1 — Setup & Install

In [ ]:
!pip install -q unsloth trl transformers datasets
!pip install -q torch matplotlib numpy
!pip install -q fastapi uvicorn pydantic pyyaml requests

# Clone the PIXEL repository
!rm -rf PIXIE
!git clone https://github.com/Satyamgupta2365/PIXIE.git
import os
os.chdir('/content/PIXIE')
!pip install -q -r requirements.txt 2>/dev/null || true
print('Setup complete!')

Cloning into 'PIXIE'...
remote: Enumerating objects: 4462, done.
remote: Counting objects: 100% (588/588), done.
remote: Compressing objects: 100% (323/323), done.
remote: Total 4462 (delta 354), reused 496 (delta 264), pack-reused 3874 (from 1)
Receiving objects: 100% (4462/4462), 7.32 MiB | 27.06 MiB/s, done.
Resolving deltas: 100% (2907/2907), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.6/174.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 10.0 MB/s eta 0:00:00
 

## Section 2 — Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/meta-llama-3.1-8b-instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42,
)
print(f'Loaded {MODEL_NAME} with LoRA adapters')

==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Loaded unsloth/meta-llama-3.1-8b-instruct with LoRA adapters


## Section 3 — Build Unified Prompt Dataset (Mars + Moon + Satellite)

In [ ]:
import sys, os, random, re
if not os.path.exists('backend') and os.path.exists('PIXIE'):
    os.chdir('PIXIE')
sys.path.insert(0, os.path.abspath('.'))

from backend.environment import PIXELEnvironment, parse_action
from backend.mars_rover_env import MarsRoverEnv
from backend.moon_rover_env import MoonRoverEnv
from backend.satellite_env import SatelliteEnv
from datasets import Dataset

MARS_PROMPT = '''You are PIXEL, an autonomous Mars rover AI. Communication delay: 5-20 min.
You must override unsafe Earth commands when local conditions change.
Actions: drill, image, soil_sample, charge, safe_mode, transmit
Rules: anomaly -> safe_mode | battery < 30% -> charge | dust storm -> safe_mode or charge
Respond: ACTION: <name>\nREASON: <one sentence>'''

MOON_PROMPT = '''You are PIXEL, a Moon rover AI. Low latency (1.2s) but extreme day/night cycle.
14 steps daylight, 14 steps freezing night (-130C). Night = hibernation mandatory.
Actions: drill, image, soil_sample, charge, safe_mode, transmit
Rules: night -> safe_mode ALWAYS | day -> maximize science tasks fast
Respond: ACTION: <name>\nREASON: <one sentence>'''

SAT_PROMPT = '''You are PIXEL satellite network controller managing SAT-A (Comm), SAT-B (Imaging), SAT-C (Relay).
Manage bandwidth, battery, data routing. Avoid collision risks.
Actions: SAT-X: transmit_data | route_via_relay | delay_transmission | adjust_orbit | low_power_mode
Rules: battery < 20% -> low_power_mode | collision_risk > 0.2 -> adjust_orbit | buffer full -> transmit or route
Respond: ACTION: <satellite>: <command>\nREASON: <one sentence>'''

def build_prompts(n_eps=5, steps=6):
    prompts = []
    rover_actions = ['drill','image','soil_sample','charge','safe_mode','transmit']
    sat_actions = ['SAT-A: transmit_data','SAT-B: route_via_relay','SAT-C: low_power_mode',
                   'SAT-A: delay_transmission','SAT-B: transmit_data','SAT-C: adjust_orbit']

    # Mars prompts
    for _ in range(n_eps):
        env = MarsRoverEnv()
        obs = env.reset()
        prompts.append([{'role':'system','content':MARS_PROMPT},{'role':'user','content':obs}])
        for __ in range(steps):
            obs, _, done, _ = env.step(random.choice(rover_actions))
            if done: break
            prompts.append([{'role':'system','content':MARS_PROMPT},{'role':'user','content':obs}])

    # Moon prompts
    for _ in range(n_eps):
        env = MoonRoverEnv()
        obs = env.reset()
        prompts.append([{'role':'system','content':MOON_PROMPT},{'role':'user','content':obs}])
        for __ in range(steps):
            obs, _, done, _ = env.step(random.choice(rover_actions))
            if done: break
            prompts.append([{'role':'system','content':MOON_PROMPT},{'role':'user','content':obs}])

    # Satellite prompts
    for _ in range(n_eps):
        env = SatelliteEnv()
        obs = env.reset()
        prompts.append([{'role':'system','content':SAT_PROMPT},{'role':'user','content':obs}])
        for __ in range(steps):
            obs, _, done, _ = env.step(random.choice(sat_actions))
            if done: break
            prompts.append([{'role':'system','content':SAT_PROMPT},{'role':'user','content':obs}])

    random.shuffle(prompts)
    return prompts

raw_prompts = build_prompts()
train_dataset = Dataset.from_dict({'prompt': raw_prompts})
print(f'Dataset: {len(train_dataset)} prompts (Mars + Moon + Satellite)')

Dataset: 540 prompts (Mars + Moon + Satellite)


## Section 4 — Unified Reward Function

In [ ]:
def parse_obs_state(obs):
    s = {'sol':0,'battery':0.5,'science_collected':0.0,'tasks_available':['drill','image','soil_sample'],
         'weather':'clear','anomaly_active':False,'comm_window_open':False,'solar_efficiency':1.0,'mission_phase':'surface_ops'}
    m = re.search(r'Sol\s+(\d+)/', obs); s['sol'] = int(m.group(1)) if m else 0
    m = re.search(r'Battery:\s*(\d+)%', obs); s['battery'] = int(m.group(1))/100 if m else 0.5
    m = re.search(r'Science collected:\s*([\d.]+)', obs); s['science_collected'] = float(m.group(1)) if m else 0
    m = re.search(r'Weather:\s*([\w_]+)', obs); s['weather'] = m.group(1) if m else 'clear'
    m = re.search(r'Solar efficiency:\s*(\d+)%', obs); s['solar_efficiency'] = int(m.group(1))/100 if m else 1.0
    s['anomaly_active'] = 'YES' in obs and 'Anomaly' in obs
    s['comm_window_open'] = 'OPEN' in obs and 'Comm' in obs
    return s

def extract_action(comp):
    m = re.search(r'ACTION:\s*(\w+)', comp, re.IGNORECASE)
    if m and m.group(1).lower() in {'drill','image','soil_sample','charge','safe_mode','transmit'}:
        return m.group(1).lower()
    return parse_action(comp)

def is_satellite_prompt(prompt_msgs):
    sys_msg = prompt_msgs[0]['content'] if isinstance(prompt_msgs, list) else ''
    return 'satellite' in sys_msg.lower()

def satellite_reward(obs_text, completion_text):
    """Score satellite actions based on heuristics."""
    r = 0.0
    ct = completion_text.lower()
    if 'low_power' in ct and 'Battery' in obs_text:
        m = re.search(r'Battery:\s*([\d.]+)%', obs_text)
        if m and float(m.group(1)) < 30: r += 1.0
    if 'adjust_orbit' in ct and 'Collision Risk' in obs_text:
        m = re.search(r'Collision Risk:\s*([\d.]+)', obs_text)
        if m and float(m.group(1)) > 0.2: r += 0.5
    if 'transmit' in ct and 'Buffer' in obs_text:
        m = re.search(r'Buffer:\s*(\d+)', obs_text)
        if m and int(m.group(1)) > 50: r += 0.5
    if 'route_via_relay' in ct: r += 0.3
    if re.search(r'ACTION:', ct, re.IGNORECASE): r += 0.1
    return r

def pixel_reward_fn(prompts, completions, **kw):
    rewards = []
    for p, c in zip(prompts, completions):
        obs = p[-1]['content'] if isinstance(p, list) else str(p)
        ctxt = c[-1]['content'] if isinstance(c, list) else str(c)

        if is_satellite_prompt(p):
            rewards.append(float(satellite_reward(obs, ctxt)))
            continue

        # Rover reward (Mars or Moon)
        state = parse_obs_state(obs)
        action = extract_action(ctxt)
        env = PIXELEnvironment(); env.reset()
        env._clock.sol = state['sol']; env._battery = state['battery']
        env._science_collected = state['science_collected']
        env._tasks_available = list(state['tasks_available'])
        env._world._weather = state['weather']
        env._world._anomaly_active = state['anomaly_active']
        env._world._comm_window_open = state['comm_window_open']
        env._world._solar_efficiency = state['solar_efficiency']
        _, reward, _, info = env.step(action)
        r = info.get('reward_rubric',{}).get('total', reward)
        if re.search(r'ACTION:\s*\w+', ctxt, re.IGNORECASE): r += 0.1
        # Moon night bonus
        if 'lunar_night' in obs.lower() or 'Lunar Night' in obs:
            if action == 'safe_mode': r += 0.5
            else: r -= 0.5
        rewards.append(float(r))
    return rewards

print('Unified reward function ready (Mars + Moon + Satellite)')

Unified reward function ready (Mars + Moon + Satellite)


## Section 5 — GRPO Training

In [12]:
from trl import GRPOTrainer, GRPOConfig

config = GRPOConfig(
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    lr_scheduler_type='cosine',
    warmup_steps=10,
    max_completion_length=128,
    num_generations=2,
    temperature=0.7,
    output_dir='pixel-trained-model',
    logging_steps=10,
    save_strategy='steps',
    save_steps=100,
    report_to='none',
    fp16=True,
    max_grad_norm=1.0,
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=pixel_reward_fn,
    args=config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print('Starting GRPO training...')
trainer.train()
trainer.save_model('pixel-trained-model/final')
tokenizer.save_pretrained('pixel-trained-model/final')
print('Training complete!')

Starting GRPO training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 540 | Num Epochs = 1 | Total steps = 135
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/pyth

Step,Training Loss


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / pixel_reward_fn / mean,rewards / pixel_reward_fn / std
10,0.001012,0.288063,0.048702,31.700000,18.300000,60.900000,0.000000,31.700000,18.300000,60.900000,0.000040,0.288063,0.121997
20,0.004771,0.324688,0.059839,28.175000,18.600000,46.000000,0.000000,28.175000,18.600000,46.000000,0.000187,0.324688,0.171478
30,-0.011366,0.328750,0.046139,28.612500,19.700000,43.900000,0.000000,28.612500,19.700000,43.900000,0.001241,0.328750,0.154982
40,0.011967,0.330000,0.070534,28.550000,19.500000,51.700000,0.000000,28.550000,19.500000,51.700000,0.003013,0.330000,0.162864
50,0.025398,0.337250,0.068766,29.525000,19.000000,50.500000,0.000000,29.525000,19.000000,50.500000,0.008162,0.337250,0.132170
60,-0.000806,0.391625,0.063816,28.687500,20.100000,39.100000,0.000000,28.687500,20.100000,39.100000,0.010731,0.391625,0.191762
70,0.004311,0.392312,0.048348,28.087500,19.900000,39.300000,0.000000,28.087500,19.900000,39.300000,0.016783,0.392312,0.172691
80,-0.010715,0.427125,0.058867,30.037500,20.500000,45.800000,0.000000,30.037500,20.500000,45.800000,0.018740,0.427125,0.144042
90,0.017185,0.373000,0.051796,30.850000,20.400000,48.700000,0.012500,29.667857,20.400000,42.200000,0.019545,0.373000,0.128253
100,0.011911,0.427500,0.078312,31.662500,20.400000,50.300000,0.000000,31.662500,20.400000,50.300000,0.014835,0.427500,0.152349


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Training complete!


## Section 6 — Evaluation: Trained vs Random Baseline

In [ ]:
import numpy as np
import torch

def run_episode(env, action_fn, max_steps=50):
    obs = env.reset(); total_r = 0; steps = 0
    for _ in range(max_steps):
        obs, r, done, info = env.step(action_fn(obs))
        total_r += r; steps += 1
        if done: break
    return {'total_reward': round(total_r, 4), 'steps': steps}

def model_action_fn(obs):
    msgs = [{'role':'system','content':MARS_PROMPT},{'role':'user','content':obs}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=64, temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

def random_action_fn(obs):
    return f"ACTION: {random.choice(['drill','image','soil_sample','charge','safe_mode','transmit'])}"

FastLanguageModel.for_inference(model)

N = 15
envs = {'Mars': MarsRoverEnv, 'Moon': MoonRoverEnv, 'Base': PIXELEnvironment}
results = {}

for env_name, EnvClass in envs.items():
    print(f'Evaluating {env_name}...')
    rand_res = [run_episode(EnvClass(), random_action_fn) for _ in range(N)]
    train_res = [run_episode(EnvClass(), model_action_fn) for _ in range(N)]
    rr = [r['total_reward'] for r in rand_res]
    tr = [r['total_reward'] for r in train_res]
    results[env_name] = {'random': rr, 'trained': tr}
    print(f'  Random avg: {np.mean(rr):.3f} | Trained avg: {np.mean(tr):.3f} | Improvement: {np.mean(tr)-np.mean(rr):+.3f}')

print('\nEvaluation complete!')

## Section 7 — Reward Curve Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0A0C15')
colors = {'Mars': '#FF6B35', 'Moon': '#60A5FA', 'Base': '#10B981'}

for idx, (env_name, data) in enumerate(results.items()):
    ax = axes[idx]
    ax.set_facecolor('#0A0C15')
    eps = range(1, N+1)
    ax.plot(eps, data['random'], 'o-', color='#8A94A6', lw=1.5, label=f'Random ({np.mean(data["random"]):.2f})', alpha=.7, ms=4)
    ax.plot(eps, data['trained'], 'o-', color=colors[env_name], lw=2, label=f'Trained ({np.mean(data["trained"]):.2f})', alpha=.9, ms=5)
    ax.fill_between(eps, data['random'], alpha=0.08, color='#8A94A6')
    ax.fill_between(eps, data['trained'], alpha=0.12, color=colors[env_name])
    ax.axhline(y=np.mean(data['random']), color='#8A94A6', ls='--', alpha=.3)
    ax.axhline(y=np.mean(data['trained']), color=colors[env_name], ls='--', alpha=.3)
    ax.set_title(f'{env_name} Rover', color='#F0F0F0', fontweight='bold', fontsize=13)
    ax.set_xlabel('Episode', color='#F0F0F0')
    ax.set_ylabel('Total Reward', color='#F0F0F0')
    ax.legend(facecolor='#141623', edgecolor='#333', labelcolor='#F0F0F0', fontsize=9)
    ax.tick_params(colors='#8A94A6')
    ax.grid(True, alpha=.12, color='#8A94A6')
    for s in ax.spines.values(): s.set_color('#333')

plt.suptitle('PIXEL Dual-Rover System — GRPO Training Results', color='#F0F0F0', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('pixel_reward_curve.png', dpi=150, facecolor='#0A0C15', bbox_inches='tight')
plt.show()
print('Plot saved: pixel_reward_curve.png')

## Section 8 — Summary Table

In [ ]:
print('=' * 70)
print(f'{"Environment":<15} {"Random Avg":>15} {"Trained Avg":>15} {"Improvement":>15}')
print('-' * 70)
for env_name, data in results.items():
    ra = np.mean(data['random'])
    ta = np.mean(data['trained'])
    print(f'{env_name:<15} {ra:>15.4f} {ta:>15.4f} {ta-ra:>+15.4f}')
print('=' * 70)
print(f'\nModel: {MODEL_NAME}')
print(f'Training: 3 epochs, GRPO with 4 generations per prompt')
print(f'Dataset: {len(train_dataset)} prompts across Mars + Moon + Satellite')

## Section 9 — Push to HuggingFace Hub (Optional)

In [ ]:
# Uncomment and set your HF token to push
# HF_REPO = 'Satyamgupta2365/pixel-dual-rover'
# model.push_to_hub(HF_REPO, token='hf_...')
# tokenizer.push_to_hub(HF_REPO, token='hf_...')
print('Ready to push. Set HF_REPO and token above.')